In [1]:
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse
import tldextract


In [2]:
df = pd.read_csv('malicious_phish.csv')
df.head()

,url,type
0,br-icloud.com.br,phishing
1,mp3raid.com/music/krizz_kaliko.html,benign
2,bopsecrets.org/rexroth/cr/1.htm,benign
3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,http://adventure-nicaragua.net/index.php?optio...,defacement


### Lexical features

In [3]:
df["url_length"] = df["url"].str.len()
df["domain"] = df["url"].apply(lambda x: urlparse(x).netloc)
df["path"] = df["url"].apply(lambda x: urlparse(x).path)
df["domain_length"] = df["domain"].str.len()
df["path_length"] = df["path"].str.len()
df["num_dots"] = df["url"].str.count(r"\.")
df["num_hyphens"] = df["url"].str.count("-")
df["num_underscores"] = df["url"].str.count("_")
df["num_slashes"] = df["url"].str.count("/")
df["num_digits"] = df["url"].str.count(r"\d")
df["num_letters"] = df["url"].str.count(r"[a-zA-Z]")
df["num_special_chars"] = df["url"].str.count(r"[^a-zA-Z0-9]")
ip_pattern = r"http[s]?://\d+\.\d+\.\d+\.\d+"
df["has_ip"] = df["url"].str.contains(ip_pattern, regex=True).astype(int)
df["has_at_symbol"] = df["url"].str.contains("@").astype(int)
df["has_double_slash_redirect"] = (
    df["url"]
    .str.replace("https://", "", regex=False)
    .str.replace("http://", "", regex=False)
    .str.contains("//")
    .astype(int)
)
df["has_https_token_in_domain"] = df["domain"].str.contains("https").astype(int)


In [4]:

#* these are to see the keywords
suspicious_words = [
    "login","verify","secure","account",
    "update","bank","confirm","free","bonus","signin","expired","ebayisapi","banking"
]

pattern = "|".join(suspicious_words)

df["has_suspicious_word"] = (
    df["url"].str.lower().str.contains(pattern).astype(int)
)

In [5]:

#* subdomains
import tldextract

def count_subdomains(url):
    ext = tldextract.extract(url)
    if ext.subdomain == "":
        return 0
    return ext.subdomain.count(".") + 1

df["num_subdomains"] = df["url"].apply(count_subdomains)

In [6]:

#* Entropy (Strong Feature)
def shannon_entropy(string):
    prob = [float(string.count(c)) / len(string) for c in dict.fromkeys(list(string))]
    return -sum([p * np.log2(p) for p in prob])

df["domain_entropy"] = df["domain"].apply(lambda x: shannon_entropy(x) if len(x) > 0 else 0)

In [7]:

#* i am only going to convert 10K of the rows since it's too large (600k)
df_10k = df.head(10_000)

In [8]:
# remove if exist so it keeps updating 

from pathlib import Path

if Path("../data/lexical_features.csv").exists() & Path("../data/Full_lexical_features.csv").exists():
    Path("../data/lexical_features.csv").unlink()
    Path("../data/Full_lexical_features.csv").unlink()   

df_10k.to_csv(Path("../data/lexical_features.csv"), index=False)
df.to_csv(Path("../data/Full_lexical_features.csv"),index=False)

In [ ]:
df_lexical = pd.read_csv('../data/lexical_features.csv')
df_lexical.head()

,url,type,url_length,domain,path,domain_length,path_length,num_dots,num_hyphens,num_underscores,...,num_digits,num_letters,num_special_chars,has_ip,has_at_symbol,has_double_slash_redirect,has_https_token_in_domain,has_suspicious_word,num_subdomains,domain_entropy
0,br-icloud.com.br,phishing,16,NaN,br-icloud.com.br,0,16,2,1,0,...,0,13,3,0,0,0,0,0,0,0.000000
1,mp3raid.com/music/krizz_kaliko.html,benign,35,NaN,mp3raid.com/music/krizz_kaliko.html,0,35,2,0,1,...,1,29,5,0,0,0,0,0,0,0.000000
2,bopsecrets.org/rexroth/cr/1.htm,benign,31,NaN,bopsecrets.org/rexroth/cr/1.htm,0,31,2,0,0,...,1,25,5,0,0,0,0,0,0,0.000000
3,http://www.garage-pirenne.be/index.php?option=...,defacement,88,www.garage-pirenne.be,/index.php,21,10,3,1,2,...,7,63,18,0,0,0,0,0,1,3.308751
4,http://adventure-nicaragua.net/index.php?optio...,defacement,235,adventure-nicaragua.net,/index.php,23,10,2,1,1,...,22,199,14,0,0,0,0,0,0,3.501398


: 